# Decision Tree — ID3 (Iterative Dichotomiser 3)

ID3 (Quinlan, 1986) builds a tree top-down for **categorical** features. At each node it picks the
attribute with the highest **information gain** and creates one branch per attribute value.

**Entropy** of a label set $S$ with class proportions $p_k$:
$$H(S) = -\sum_k p_k \log_2 p_k$$

**Information gain** of splitting $S$ on attribute $A$:
$$IG(S, A) = H(S) - \sum_{v\in \text{values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

**Algorithm**
1. If all samples have the same class → leaf with that class.
2. If no attributes remain → leaf with the majority class.
3. Otherwise choose $A^* = \arg\max_A IG(S, A)$, create a branch for each value $v$ of $A^*$ and recurse on $S_v$ with $A^*$ removed.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

## Entropy and information gain

In [ ]:
def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))


def information_gain(x_col, y):
    weighted_child_entropy = 0.0
    for v in np.unique(x_col):
        subset = y[x_col == v]
        weighted_child_entropy += len(subset) / len(y) * entropy(subset)
    return entropy(y) - weighted_child_entropy


p = np.linspace(0.001, 0.999, 200)
plt.plot(p, -p * np.log2(p) - (1 - p) * np.log2(1 - p))
plt.xlabel("P(class = 1)"); plt.ylabel("entropy (bits)"); plt.title("Binary entropy"); plt.show()

## The classic *Play Tennis* dataset

In [ ]:
feature_names = ["Outlook", "Temperature", "Humidity", "Wind"]
data = np.array([
    ["Sunny",    "Hot",  "High",   "Weak",   "No"],
    ["Sunny",    "Hot",  "High",   "Strong", "No"],
    ["Overcast", "Hot",  "High",   "Weak",   "Yes"],
    ["Rain",     "Mild", "High",   "Weak",   "Yes"],
    ["Rain",     "Cool", "Normal", "Weak",   "Yes"],
    ["Rain",     "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Strong", "Yes"],
    ["Sunny",    "Mild", "High",   "Weak",   "No"],
    ["Sunny",    "Cool", "Normal", "Weak",   "Yes"],
    ["Rain",     "Mild", "Normal", "Weak",   "Yes"],
    ["Sunny",    "Mild", "Normal", "Strong", "Yes"],
    ["Overcast", "Mild", "High",   "Strong", "Yes"],
    ["Overcast", "Hot",  "Normal", "Weak",   "Yes"],
    ["Rain",     "Mild", "High",   "Strong", "No"],
])
X, y = data[:, :-1], data[:, -1]
print(X.shape, "classes:", np.unique(y, return_counts=True))

In [ ]:
print(f"Entropy of the full dataset: {entropy(y):.4f}\n")
for i, name in enumerate(feature_names):
    print(f"IG(S, {name:<11}) = {information_gain(X[:, i], y):.4f}")

**Outlook** has the highest gain, so it becomes the root. Now the full recursive algorithm:

In [ ]:
class ID3:
    """ID3 decision tree for categorical features.

    Internal nodes are dicts {"feature", "majority", "children": {value: subtree}};
    leaves are plain class labels.
    """

    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    def fit(self, X, y, feature_names=None):
        X, y = np.asarray(X), np.asarray(y)
        self.feature_names = feature_names or [f"x{i}" for i in range(X.shape[1])]
        self.tree = self._build(X, y, list(range(X.shape[1])), depth=0)
        return self

    def _build(self, X, y, features, depth):
        classes, counts = np.unique(y, return_counts=True)
        majority = classes[np.argmax(counts)]

        if len(classes) == 1 or not features or (self.max_depth is not None and depth >= self.max_depth):
            return majority

        gains = [information_gain(X[:, f], y) for f in features]
        best = features[int(np.argmax(gains))]
        remaining = [f for f in features if f != best]

        node = {"feature": best, "majority": majority, "children": {}}
        for v in np.unique(X[:, best]):
            mask = X[:, best] == v
            node["children"][v] = self._build(X[mask], y[mask], remaining, depth + 1)
        return node

    def _predict_one(self, x):
        node = self.tree
        while isinstance(node, dict):
            # unseen attribute value -> fall back to majority class at this node
            node = node["children"].get(x[node["feature"]], node["majority"])
        return node

    def predict(self, X):
        return np.array([self._predict_one(x) for x in np.asarray(X)])

    def print_tree(self, node=None, indent=""):
        node = self.tree if node is None else node
        if not isinstance(node, dict):
            print(indent + "-> " + str(node))
            return
        for value, child in node["children"].items():
            print(f"{indent}[{self.feature_names[node['feature']]} = {value}]")
            self.print_tree(child, indent + "    ")

In [ ]:
tree = ID3().fit(X, y, feature_names)
tree.print_tree()
print("\nTraining accuracy:", np.mean(tree.predict(X) == y))

### Predicting new days

In [ ]:
new_days = np.array([
    ["Sunny", "Cool", "High", "Strong"],
    ["Overcast", "Hot", "High", "Weak"],
    ["Rain", "Hot", "Normal", "Weak"],
    ["Sunny", "Mild", "Normal", "Weak"],
])
for day, pred in zip(new_days, tree.predict(new_days)):
    print(dict(zip(feature_names, day.tolist())), "->", pred)

## Visualising the tree
A small recursive plotter with matplotlib.

In [ ]:
def plot_id3(model, figsize=(11, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    def n_leaves(node):
        return 1 if not isinstance(node, dict) else sum(n_leaves(c) for c in node["children"].values())

    def draw(node, x_left, x_right, y, parent_xy=None, edge_label=""):
        x = (x_left + x_right) / 2
        if parent_xy:
            ax.plot([parent_xy[0], x], [parent_xy[1], y], "k-", lw=1)
            ax.text((parent_xy[0] + x) / 2, (parent_xy[1] + y) / 2, edge_label,
                    fontsize=9, color="darkblue", ha="center", backgroundcolor="w")
        if isinstance(node, dict):
            ax.text(x, y, model.feature_names[node["feature"]], ha="center", va="center",
                    bbox=dict(boxstyle="round", fc="lightyellow"), fontsize=11)
            total, start = n_leaves(node), x_left
            for v, child in node["children"].items():
                width = (x_right - x_left) * n_leaves(child) / total
                draw(child, start, start + width, y - 1, (x, y), v)
                start += width
        else:
            ax.text(x, y, str(node), ha="center", va="center", fontsize=11,
                    bbox=dict(boxstyle="round", fc="lightgreen" if node == "Yes" else "salmon"))

    draw(model.tree, 0, 1, 0)
    ax.set_ylim(-3.5, 0.5)
    plt.title("ID3 tree for Play Tennis"); plt.show()


plot_id3(tree)

## Limitations of ID3
* Only categorical attributes (continuous ones must be discretised first).
* Information gain is biased towards attributes with many values (e.g. an ID column gives perfect gain).
* No pruning → prone to overfitting; no handling of missing values.

These are addressed by **C4.5** (next notebook).